# 📚 LangChain ile Retrieval Augmented Generation'a Giriş 🦜🔗

Bu not defterinde LangChain kullanarak Retrieval Augmented Generation'ı nasıl kullanacağınızı öğreneceksiniz.

Kendi belgelerimiz hakkında sorular sormak için bir LLM kullanacağız!

## ⚙️ Kurulum

👉 Temel kütüphaneleri içe aktarmak için aşağıdaki hücreyi çalıştırın.

In [8]:
%load_ext autoreload
%autoreload 2
import os
from pprint import pprint
from IPython.display import Markdown

👉 API anahtarımızı tekrar yüklemek için aşağıdaki hücreyi çalıştırın:

In [9]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

## 📚 Neden RAG?

Bir LLM kendi başına, öğrendiği her şey hakkında sorulara yanıt verebilir.

Bunun birkaç dezavantajı vardır:
- Eğitim verileri geçmişten gelir ve en son verilerle güncellenmez.
- Sadece eğitim aldığı verileri bilir.

Bir LLM'yi kendi verilerimizle çalışması için kullanmak istiyoruz. İşte bu noktada RAG (Retrieval-Augmented Generation) devreye girer.

1. **Retrieval-Augmented Generation (RAG)**, gerçek doğruluğu artırmak için bir dil modelini belge alıcı ile birleştirir.
2. **İlgili dış belgeleri alır** (örneğin, bilgi tabanından) yanıtlar üretmeden önce.
3. **Dil modeli hem istemi hem de alınan bağlamı kullanarak** daha bilgili ve temelli çıktılar üretir.

## 🇪🇺 Bağlam

Bu meydan okumada, Avrupa Parlamentosu'ndan belgelerle çalışacağız.

Bir gazeteci olduğunuzu ve Avrupa Parlamentosu'nun genel kurul oturumları sırasında belirli bir konu hakkında neler söylendiğini öğrenmek istediğinizi düşünün. Bu oturumlar yılda 12 kez Strasbourg'da gerçekleşir ve 4 gün sürer. Oturumların transkriptleri EP'nin web sitesinde mevcuttur.

Kesinlikle tüm bu transkriptleri karıştırmak istemezsiniz. O halde, hayatımızı kolaylaştırmak için RAG'ı kullanalım!

Bu, her zaman test etmek için yepyeni veriler alabileceğimiz için çalışmak üzere iyi verilerdir.

## 📘 Verileri alalım

1. [EP'nin web sitesine](https://www.europarl.europa.eu/plenary/en/debates-video.html) gidin. 
1. Bu sizi en son genel kurul oturumuna yönlendirecektir.
1. İlk tarihin altında, "▶️ Verbatim reports HTML" bölümünde `HTML`'e tıklayın.
1. Sayfanın sonuna kaydırın ve alttaki PDF dosyasını indirin.
1. Dosyayı `data` klasörüne kaydedin.

Bir belgeyle başlayacağız, ancak daha sonrası için diğer birkaç günün aynısını şimdiden indirebilirsiniz.

Belgeye bir göz atın. Kaç sayfası var? Belge hakkında bir fikir edinmek için hızlıca belgede gezinin.

## 🔢 Belgeleri gömme

Belgeleri gömmek, tüm belgeleri veya belge parçalarını vektörlere çevirmek anlamına gelir.

LangChain🦜🔗 yine çok yardımcı olacak.

Bir gömme aracı (embedder) başlatalım ve deneyelim. LLM olarak Gemini kullandığımız için, Google'ın metin gömme araçlarında kalalım.

In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

👉 Basit bir metin parçasını gömmek için gömme aracının `.embed_query()` metodunu deneyin.

In [11]:
!pip install langchain-google-genai

In [12]:
# Embed a text like "What is the capital of France?" and save it to a variable `sample_embedding`

sample_embedding = embeddings.embed_query("What is the capital of France?")

print("Metin başarıyla gömüldü ve 'sample_embedding' değişkenine aktarıldı.")
print(f"Vektör Boyutu: {len(sample_embedding)}")
print(f"İlk 5 Sayısal Değer: {sample_embedding[:5]}")

Metin başarıyla gömüldü ve 'sample_embedding' değişkenine aktarıldı.
Vektör Boyutu: 3072
İlk 5 Sayısal Değer: [-0.032554302364587784, 0.013054960407316685, 0.015067406930029392, -0.07128717750310898, -0.03106236644089222]


👉 Bu `sample_embedding`'i keşfetmek için zaman ayırın. Nasıl görünüyor? Tipi nedir? Gömme boyutu nedir?

In [13]:
print(f" Veri Tipi: {type(sample_embedding)}")

print(f" Gömme Boyutu (Boyut Sayısı): {len(sample_embedding)}")

print(f" İlk 3 Elemanın Tipi: {[type(x) for x in sample_embedding[:3]]}")

print("\n Vektörün Görünümü (Örnek Kesit):")
print(f"[{sample_embedding[0]}, {sample_embedding[1]}, {sample_embedding[2]}, ..., {sample_embedding[-2]}, {sample_embedding[-1]}]")

 Veri Tipi: <class 'list'>
 Gömme Boyutu (Boyut Sayısı): 3072
 İlk 3 Elemanın Tipi: [<class 'float'>, <class 'float'>, <class 'float'>]

 Vektörün Görünümü (Örnek Kesit):
[-0.032554302364587784, 0.013054960407316685, 0.015067406930029392, ..., -0.01693323813378811, -0.022551335394382477]


## 💾 PDF'den gerçek verilerimizi yükle

Artık bir gömmenin nasıl göründüğünü biliyoruz, gerçek verilerimizle çalışmanın zamanı geldi.

👉 [LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/document_loaders/index#pdfs) gidin ve PyPDF kullanarak bir PDF'yi nasıl yükleyebileceğinizi öğrenin.

👉 Sonra devam edin ve daha önce indirdiğiniz PDF'lerden birini yükleyin.

In [14]:
%pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [15]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "data/CRE-2026-04-27.pdf"
loader = PyPDFLoader(file_path)

# Dosya artık tamamen dolu olduğu için sorunsuz çalışacak
pages = loader.load()

print(f"📄 Toplam Sayfa Sayısı: {len(pages)}")

📄 Toplam Sayfa Sayısı: 131


👉 `pages`'i keşfedin:
- Veri tipi nedir?
- Kaç sayfanız var?
- Bir sayfanın tipi nedir?
- Bir sayfanın içeriğine nasıl erişebilirsiniz?
- Tam belgenin kaç karakteri var?
- Bir sayfanın `metadata`'sında neler var?

In [16]:
print(f" 1. 'pages' Veri Tipi: {type(pages)}")

total_pages = len(pages)
print(f" 2. Toplam Sayfa Sayısı: {total_pages}")

single_page = pages[0]
print(f" 3. Bir Sayfanın Tipi: {type(single_page)}")

print(f" 4. Sayfa İçeriğine Erişim Örneği (İlk 150 Karakter):\n{single_page.page_content[:150]}")

total_characters = sum(len(page.page_content) for page in pages)
print(f" 5. Tam Belgenin Toplam Karakter Sayısı: {total_characters:,}")

print(f" 6. Bir Sayfanın Metadata İçeriği:\n{single_page.metadata}")

 1. 'pages' Veri Tipi: <class 'list'>
 2. Toplam Sayfa Sayısı: 131
 3. Bir Sayfanın Tipi: <class 'langchain_core.documents.base.Document'>
 4. Sayfa İçeriğine Erişim Örneği (İlk 150 Karakter):
2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ 
 5. Tam Belgenin Toplam Karakter Sayısı: 413,397
 6. Bir Sayfanın Metadata İçeriği:
{'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '489873', 'dmxml.render.traceid': '6a048177c164c099cb46b1e4803406df', 'uid': 'eu.europa.europarl-DIN1-2026-0000138944_01.00-xm-01.00_text-xml', 'source': 'data/CRE-2026-04-27.pdf', 'total_pages': 131, 'page': 0, 'page_label': '1'}


## ✂️ Verilerimizi böl

Tam belgemiz gömülmek için çok uzun. Metin gömme aracımız 2.048 tokena kadar giriş alabilir. Gemini modelleri için bu yaklaşık 8.196 karakterdir (token başına 4 karakter).

Bu yüzden belgemizi daha küçük parçalara bölmek istiyoruz.

Zaten çalışabileceğimiz bir dizi sayfamız var. Ama sayfa sonları biraz keyfi: genellikle cümlenin ortasında görünürler.

Ayrıca, sayfalar arasında örtüşme yoktur. Bu yüzden bir sayfanın ilk satırı önceki tüm bağlamı kaçırır. Tam metni biraz örtüşmeyle bölmek daha iyidir.

İlk olarak, PDF'yi tekrar yükleyeceğiz, bu sefer bölmeden.

In [17]:
loader = PyPDFLoader(file_path, mode='single')
pdf = loader.load()
pdf_text = pdf[0].page_content
len(pdf_text)

413657

Artık tüm PDF'imizi tek bir belge olarak aldığımıza göre, onu daha akıllı bir şekilde parçalara bölebiliriz.

👉 Yine, ["Özyinelemeli olarak bölme" konusundaki LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) gidin ve `pdf` _belgelerimizi_ parçalara (LangChain'de `documents` olarak adlandırılır) nasıl böleceğinizi öğrenin.

2_000 karakter (bizim durumumuzda yaklaşık yarım sayfa) parçalara 400 örtüşmeyle bölün. İsterseniz diğer değerlerle deneyebilirsiniz.

`RecursiveCharacterTextSplitter`'ın `.split_documents()` metodunu kullanın: bu metod giriş olarak bir belge alır ve bölünmüş belgeler çıktısı verir.

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400,
    length_function=len
)

chunked_docs = text_splitter.split_documents(pages)

print(f"Parçalama işlemi tamamlandı")
print(f"Orijinal Sayfa Sayısı: {len(pages)}")
print(f"Oluşan Toplam Parça (Chunk) Sayısı: {len(chunked_docs)}")

Parçalama işlemi tamamlandı
Orijinal Sayfa Sayısı: 131
Oluşan Toplam Parça (Chunk) Sayısı: 284


👉 `all_splits`'i inceleyin:
- Veri tipi nedir?
- Kaç bölümünüz var?
- Bir bölümün tipi nedir?
- Bir bölümün içeriğine nasıl erişebilirsiniz?
- Şimdi toplamda kaç karakterimiz var?
- Bir bölümün `metadata`'sında neler var?

In [19]:
all_splits = chunked_docs

print(f"📋 1. 'all_splits' Veri Tipi: {type(all_splits)}")
total_chunks = len(all_splits)

print(f"🔢 2. Toplam Bölüm (Chunk) Sayısı: {total_chunks}")
single_split = all_splits[0]

print(f"🧱 3. Bir Bölümün Tipi: {type(single_split)}")
print(f"📝 4. Bölüm İçeriğine Erişim Örneği:\n{single_split.page_content[:150]}")

total_split_characters = sum(len(split.page_content) for split in all_splits)
print(f"🔤 5. Bölünmüş Haldeki Toplam Karakter Sayısı: {total_split_characters:,}")
print(f"🗂️ 6. Bir Bölümün Metadata İçeriği:\n{single_split.metadata}")

📋 1. 'all_splits' Veri Tipi: <class 'list'>
🔢 2. Toplam Bölüm (Chunk) Sayısı: 284
🧱 3. Bir Bölümün Tipi: <class 'langchain_core.documents.base.Document'>
📝 4. Bölüm İçeriğine Erişim Örneği:
2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ 
🔤 5. Bölünmüş Haldeki Toplam Karakter Sayısı: 467,467
🗂️ 6. Bir Bölümün Metadata İçeriği:
{'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '489873', 'dmxml.render.traceid': '6a048177c164c099cb46b1e4803406df', 'uid': 'eu.europa.europarl-DIN1-2026-0000138944_01.00-xm-01.00_text-xml', 'source': 'data/CRE-2026-04-27.pdf', 'total_pages': 131, 'page': 0, 'page_label': '1'}


## 🗄️ Her şeyi bir araya getir: belgelerimizi gömme ve vektör deposunda sakla

Elimizde şunlar var:
- Bir gömme aracı
- Veriyi yüklemek için bir yükleyici
- Belgemizi belgelere bölmek için bir metin bölücü

Neyi kaçırıyoruz?

Belgelerimizi gömebiliriz, ama onları bir yerde saklamak istiyoruz. İşte burada vektör deposu devreye girer: şunları saklamamıza olanak sağlar:
- belgeyi (parçayı),
- onun gömmesini,
- meta verilerini.

Sonraki adımda belgeleri verimli bir şekilde alabilecek olacağız.

👉 Bir `InMemoryVectorStore` nasıl oluşturabileceğinizi görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

In [20]:
%pip install --upgrade --quiet langchain-google-genai langchain-core

Note: you may need to restart the kernel to use updated packages.


In [22]:
# Import the necessary libraries
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",
    project="gen-lang-client-0713333846",
    location="us-central1"
)
# Create an in-memory vector store using the embedder `embeddings` we created earlier
vector_store = InMemoryVectorStore(embedding=embeddings)

# Add the `all_splits` to the vector store and store the result in a variable called `document_ids`
document_ids = vector_store.add_documents(documents=all_splits)

print("Vektör deposu başarıyla oluşturuldu ve belgeler mühürlendi!")
print(f"Üretilen Toplam Belge ID Sayısı: {len(document_ids)}")

Vektör deposu başarıyla oluşturuldu ve belgeler mühürlendi!
Üretilen Toplam Belge ID Sayısı: 284


In [23]:
# Have a look at the first 3 document IDs

print(f"İlk 3 Belge ID Örneği: {document_ids[:3]}")

İlk 3 Belge ID Örneği: ['aba4bc37-374c-4806-a2a4-c40e585b823b', 'f7072082-9b61-4ce7-b716-771a7cb1db5f', '80b6191c-5df1-48b8-a464-2dce843af065']


In [24]:
# Use the vector store's `get_by_ids` method. You have to give it a list of document IDs.
sample_ids = document_ids[:2]

retrieved_docs = vector_store.get_by_ids(sample_ids)

print(f"İstenen ID Sayısı: {len(sample_ids)} | Geri Getirilen Döküman Sayısı: {len(retrieved_docs)}")
print("\n--- İlk Geri Getirilen Dökümanın İçeriği (İlk 200 Karakter) ---")
print(retrieved_docs[0].page_content[:200])

İstenen ID Sayısı: 2 | Geri Getirilen Döküman Sayısı: 2

--- İlk Geri Getirilen Dökümanın İçeriği (İlk 200 Karakter) ---
2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FOR


👉 Bir vektör deposundaki belgenin içeriğine ve meta verilerine nasıl erişebilirsiniz?

In [25]:
sample_doc = retrieved_docs[0]

pure_text = sample_doc.page_content

meta_info = sample_doc.metadata

print("BELGE İÇERİĞİ:\n", pure_text[:300], "...\n")
print("META VERİLERİ:\n", meta_info)

BELGE İÇERİĞİ:
 2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG V ...

META VERİLERİ:
 {'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '489873', 'dmxml.render.traceid': '6a048177c164c099cb46b1e4803406df', 'uid': 'eu.europa.europarl-DIN1-2026-0000138944_01.00-xm-01.00_text-xml', 'source': 'data/CRE-2026-04-27.pdf', 'total_pages': 131, 'page': 0, 'page_label': '1'}


## 🔎 Benzer belgeleri almak için vektör deposunu kullan

Artık belgeleri gömleğe çevirdiğimize göre, benzer belgeleri almak için vektör deposunu kullanabiliriz.

👉 Bunun nasıl çalıştığını görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

Bir sorgu kullanın, örneğin "Tarım politikası üzerine tartışmayı özetle.", ve en benzer belgeleri bulun. Ayrıca alınacak belge sayısını da belirtebilirsiniz.

In [26]:
# Save your question into a variable called `query`

query = "Tarım politikası üzerine tartışmayı özetle."

# Use the vector store to find similar documents to the query. Store the result in a variable called `retrieved_docs`

retrieved_docs = vector_store.similarity_search(query, k=3)

print(f"Sorgu: '{query}'")
print(f"Başarıyla {len(retrieved_docs)} adet benzer belge geri getirildi!")
print(f"Gelen ilk parçanın sayfa numarası: {retrieved_docs[0].metadata['page']}")

Sorgu: 'Tarım politikası üzerine tartışmayı özetle.'
Başarıyla 3 adet benzer belge geri getirildi!
Gelen ilk parçanın sayfa numarası: 16


Bu, RAG'ın sözde "Alma" (Retrieval) kısmını tamamlar: artık sorgumuza en benzer belgeleri bulabiliriz.

Çalışmanın çoğu artık tamamlandı!

## 💬 Sorumuza bir cevap üret

Şimdiye kadar benzer belgeleri almamızı sağlamak için sadece bir **gömme modeli** kullandık.

Şimdi, sorumuzla bir cevap almak için üretici bir LLM kullanacağız: ona aldığımız belgeler ve sorumuzla besleyeceğiz.

Bunu yapmanın en temel yolu tüm girdilerimizi birbirine bağlamak, sorumuzla eklemek ve sonucu görmek olacaktır.

Bir deneyelim.

👉 İlk olarak önceki meydan okumalarda olduğu gibi bir LLM başlatın.

In [50]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gemini-2.5-flash-lite", model_provider="google_genai")

Sonra temel bir istem oluşturun:

In [51]:
prompt = '\n\n'.join([doc.page_content for doc in retrieved_docs])
prompt += "\n\n" + query

👉 Şimdi istemi kullanın:

In [52]:
response = llm.invoke(prompt)

print("YAPAY ZEKANIN CEVABI:\n")
print(response.content)

YAPAY ZEKANIN CEVABI:

A rendelkezésemre álló szövegek a mezőgazdasági politikáról szóló vitát nem ismertetik. A szövegek a Parlament ülésszakának tartalmát, valamint a szexuális bűncselekményekkel, az áldozatvédelemmel és a jogrenddel kapcsolatos képviselői nyilatkozatokat tartalmazzák.


Bu fena değil, ama modele daha fazla rehberlik vererek daha kapsamlı bir istem yazarak daha iyisini yapabiliriz.

Bunu yapan ilk kişiler biz değilmişiz ve LangChain'in bizim için önceden hazırlanmış istem kütüphanesi var.

👉 Aşağıdaki hücreyi çalıştırın ve nasıl çalıştığını anlamaya çalışın. (LangSmithMissingAPIKeyWarning hakkında bir uyarı alacaksınız, bunu görmezden gelebilirsiniz.)

In [63]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. Use the following pieces of "
    "retrieved context to answer the question. If you don't know the answer, just say "
    "that you don't know, don't try to make up an answer. Use three sentences maximum "
    "and keep the answer concise.\n\n"
    "Context: {context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

example_messages = prompt_template.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
)

print(example_messages)

messages=[HumanMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know, don't try to make up an answer. Use three sentences maximum and keep the answer concise.\n\nContext: (context goes here)\n\nQuestion: (question goes here)\n\nAnswer:", additional_kwargs={}, response_metadata={})]


LangChain'in bizim için nasıl daha kesin bir istem oluşturduğunu görüyor musunuz? Bunu RAG'ımız için kullanalım!

👉 İlk olarak, tüm alınan belgeleri iki yeni satırla ayrılmış tek bir uzun dizgiye birleştirin.

In [64]:
docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)
Markdown(docs_content)

27-04-2026  17 
HU TARTALOM 
1 - Az ülésszak folytatása .........................................................................................................27 
2 - Az ülés megnyitása ..............................................................................................................27 
3 - Az elnök nyilatkozatai .........................................................................................................27 
4 - Az előző ülésnapok jegyzőkönyveinek elfogadása .............................................................28 
5 - A Parlament tagjai ...............................................................................................................28 
6 - Mandátumvizsgálat ..............................................................................................................29 
7 - Képviselői mentelmi jog felfüggesztésére irányuló kérelmek .............................................29 
8 - A bizottságok és a küldöttségek tagjai ................................................................................29 
9 - A Parlament első olvasatát megelőző tárgyalások (az eljárási szabályzat 72. cikke) .........29 
10 - A harmadik olvasatra vonatkozó határidők meghosszabbítása (az ejárási szabályzat 76. 
cikke) .........................................................................................................................................30 
11 - Helyesbítések (az eljárási szabályzat 251. cikke) ..............................................................30 
12 - A rendes jogalkotási eljárás keretében elfogadott jogi aktusok aláírása (az eljárási 
szabályzat 81. cikke) .................................................................................................................30 
13 - Ügyrend .............................................................................................................................30

16  27-04-2026 
LT TURINYS 
1 - Sesijos pratęsimas ................................................................................................................27 
2 - Posėdžio pradžia ..................................................................................................................27 
3 - Pirmininkės pareiškimai ......................................................................................................27 
4 - Ankstesnių plenarinių posėdžių protokolų tvirtinimas ........................................................28 
5 - Parlamento sudėtis ...............................................................................................................28 
6 - Igaliojimų tikrinimas ...........................................................................................................29 
7 - Prašymai atšaukti imunitetą .................................................................................................29 
8 - Komitetų ir delegacijų sudėtis .............................................................................................29 
9 - Derybos prieš pirmąjį svarstymą Parlamente (Darbo tvarkos taisyklių 72 straipsnis) ........29 
10 - Trečiajam svarstymui numatytų terminų pratęsimas (Darbo tvarkos taisyklių 76 
straipsnis) ..................................................................................................................................30 
11 - Klaidų ištaisymai (Darbo tvarkos taisyklių 251 straipsnis) ...............................................30 
12 - Pagal įprastą teisėkūros procedūrą priimtų aktų pasirašymas (Darbo tvarkos taisyklių 81 
straipsnis) ..................................................................................................................................30 
13 - Darbų programa .................................................................................................................30

27-04-2026  75 
képvisel: erősebb áldozatvédelemre, hatékonyabb bűnüldözésre és kiszámítható jogrendre van 
szükség.  
 
Mindenkinek egyértelműen tudnia kell, hol húzódnak a határok, és mi jár büntetőjogi 
következményekkel. A beleegyezésen alapuló szabályozás pontosan erről szól.  
 
1-0156-7500 
Katri Kulmuni (Renew), kirjallinen – Käsitys seksuaalirikosten luonteesta on kehittynyt viime 
vuosikymmeninä. Aikaisemmin niitä on voitu pitää lähinnä väkivaltarikoksina, jolloin teon 
tunnusmerkistö ja myös teon törkeyden määrittely on voinut perustua lähinnä fyysisen väkivallan 
voimakkuuteen. Tuolloin ei ole ymmärretty rikoksen ja sen uhrille koituvien seurausten 
kokonaisvaltaista luonnetta. 
 
 
 
Rikosoikeus on lähtökohtaisesti kansallista ja kansalliset parlamentit säätävät rikoslait. Tietty 
minimitaso määritelmien suhteen olisi näissä törkeimmissä rikoksissa kuitenkin paikallaan. 
Suostumusperusteen määrittely raiskausten tunnusmerkistöön tulisi saada kaikissa EU-maissa 
voimaan - ja kaikki maailman maat saisivat tätä esimerkkiä seurata. 
 
1-0156-8125 
András László (PfE), írásban – Elkötelezettek vagyunk az áldozatok hatékony védelme és a 
megelőzést célzó intézkedések előmozdítása mellett, ezen törekvések azonban csak akkor 
lehetnek eredményesek, ha a szükséges intézkedések nemzeti szinten, az egyes tagállamok 
társadalmi- és jogi sajátosságaihoz igazodnak. Az egységes, központosított megközelítés 
figyelmen kívül hagyja azokat a nemzeti, kulturális és jogi különbségeket, amelyek 
kulcsfontosságúak a valódi segítségnyújtás szempontjából. Aggályosnak tartjuk, hogy a baloldal 
több ponton ideológiai elemeket, különösen a genderideológiát igyekszik beemelni a határozatba, 
és ahelyett, hogy az áldozatok támogatását erősítené, bizonytalan jogi helyzetet teremtene. Ez 
nem szolgálja az áldozatok érdekeit. A határozat meghatározó eleme a tömeges illegális 
migrációval összefüggésben növekvő nemi erőszak kérdése, miközben ez nyíltan nem merik

👉 Sonra, sorgunuz ve alınan belgelerden başlayarak bir `prompt` oluşturun. Yukarıdaki örneğe bakmayı unutmayın.

In [65]:
prompt = prompt_template.invoke(
    {"context": docs_content, "question": query}
)

👉 Son olarak az önce oluşturduğumuz `the_prompt` ile LLM modelini kullanın:

In [66]:
answer = llm.invoke(prompt)

In [67]:
Markdown(answer.content)

I am sorry, but the provided context does not contain information about a discussion on agricultural policy. The text discusses the agenda of a parliamentary session, including items like the continuation of a session, opening of a meeting, president's statements, approval of minutes, parliamentary members, examination of mandates, requests to suspend parliamentary immunity, composition of committees and delegations, negotiations before the first reading, extension of deadlines for the third reading, corrections, signing of legal acts, and the agenda. It also includes statements from Katri Kulmuni and András László concerning sexual offenses and victim protection, with discussions on consent-based regulation and national legal specificities.

🎉 İlk RAG'ımızı tamamladık: LLM kendisine sağladığımız belgelerde ***temelli*** metin üretti.

## 💾 Gömmelerimizi kalıcı hale getir

Şimdiye kadar bellekte vektör deposuyla çalıştık. Bu yüzden not defterinizi kapattığınızda, tüm gömmeleri de kaybedeceksiniz.

⚠️ Bu gömmelerin sağlayıcınızın platformunda, bu durumda Google'ın makinelerinde çalışan modeller tarafından üretildiğini unutmayın. Ve bedava çalışmazlar. 💰

Bunun gibi bir, nispeten küçük belge için maliyet düşüktür, ama hızla artar. Şimdiye kadar sadece bir günün transkriptleriyle çalıştık. Oturum başına 3 tane daha, yılda 12 oturum, birden fazla yıl var...

Bunu çözmek için sadece vektör depomuzla kalıcı bir taneyi değiştireceğiz. Bu LangChain'in avantajıdır: bileşenleri değiştirmek çok kolay.

Bellekteki vektör depomuz deneme için harikaydı, şimdi başka bir taneyle değiştireceğiz. Çok popüler bir vektör deposu olan [Chroma](https://www.trychroma.com/)'yı kullanacağız. Bunu yerel olarak çalıştırabilir ve LangChain aracılığıyla kullanabiliriz.

Tüm akışımızı yeniden oluşturacağız. Her şeyi birkaç kod hücresinde tekrar bir araya getirmeye çalışmak iyi bir alıştırmadır. Aynı zamanda her şeyi yeniden kullanılabilir koda dönüştüreceğiz.

Sonunda iki fonksiyon istiyoruz:

1. `embed_and_store()`: Başka bir oturumun transkriptini vektör veritabanımıza ekle, böylece alacağımız daha fazla veri olsun.
2. `answer()`: Vektör depomuzla farklı sorularla sorgula.

#### 1. Bir Chroma vektör deposu başlat

👉 **Veri kalıcılığıyla** (yani verileri diskteki bir dizinde saklayarak) Chroma vektör deposunun nasıl oluşturulacağını görmek için [LangChain'in belgelerine](https://python.langchain.com/docs/integrations/vectorstores/chroma/) bakın.

In [70]:
%pip install langchain-chroma

  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 3.8 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 3.7 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 3.9 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 4.0 MB/s  0:00:04m0:00:0100:01
Using cached overrides-7.7.0-py3-none-any.whl (17 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 4.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 671.5/671.5 kB 4.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 2.8 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 2.4 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41/41 [langchain-chroma][chromadb]s]-hub]k]o

In [3]:
import os
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/text-embedding-004",
    project="gen-lang-client-0713333846",
    location="us-central1"
)

vector_store = Chroma(
    collection_name="ep_plenary",
    embedding_function=embeddings,
    persist_directory="./chroma_ep_follower",
)

#### 2. `embed_and_store()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [7]:
def embed_and_store(file_path, vector_store):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store."""
    # Load the PDF file
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    
    loader = PyPDFLoader(file_path, mode='single')
    pdf=loader.load()

    # Split the pages into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2_000,
        chunk_overlap=400,
        add_start_index=True,
    )
    all_splits= text_splitter.split_documents(pdf)

    # # Add the session_date to the metadata
    # for split in all_splits:
    #     split.metadata['session_date'] = session_date

    # Add the chunks to the vector store
    document_ids = vector_store.add_documents(documents=all_splits)
    print(f"Added {len(document_ids)} documents to the vector store.")

    return document_ids

👉 Fonksiyonunuzu bir dosya veya hatta iki dosyayla deneyin:

In [10]:
file_path = "data/CRE-2026-04-27.pdf"

document_ids = embed_and_store(file_path, vector_store)

Added 259 documents to the vector store.


In [12]:
vector_store.get_by_ids(document_ids[:3])

[Document(id='dfab0e87-21bb-4dab-9693-b552189969d3', metadata={'creationdate': '', 'dmxml.render.traceid': '6a048177c164c099cb46b1e4803406df', 'creator': 'Aspose.Words', 'dmxml.render.id': '489873', 'author': 'e-Parliament@europarl.europa.eu', 'source': 'data/CRE-2026-04-27.pdf', 'start_index': 0, 'uid': 'eu.europa.europarl-DIN1-2026-0000138944_01.00-xm-01.00_text-xml', 'total_pages': 131, 'producer': 'Aspose.Words for Java 24.2.0'}, page_content='2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z

#### 3. `answer()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [19]:
def answer(query, vector_store, llm, prompt_template=None):
    """Answer a query using the vector store and the language model."""
    # Retrieve similar documents from the vector store
    retrieved_docs = vector_store.similarity_search(query, k=6)

    # Create the prompt
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # If no prompt template is provided, use the default one
    if not prompt_template:
        prompt_template = hub.pull("rlm/rag-prompt")

    prompt = prompt_template.invoke(
        {"context": docs_content, "question": query}
    )

    # Get the answer from the language model
    answer = llm.invoke(prompt)

    return answer.content

In [24]:
%pip install langchain-google-vertexai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 2.8 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 3.1 MB/s  0:00:15m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.9/934.9 kB 3.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31/31 [langchain-google-vertexai]n-google-vertexai]er]
Note: you may need to restart the kernel to use updated packages.


👉 Fonksiyonunuzu beğendiğiniz bir sorguyla deneyin:

In [1]:
query = "What is being said about international trade?"
Markdown(answer(query, vector_store, llm, prompt_template))

NameError: name 'Markdown' is not defined

🏁 Tebrikler! Artık LangChain kullanarak RAG'da ustalaştınız ve vektör deponuza daha fazla belge eklemek ve onu sorgulamak için yeniden kullanılabilir fonksiyonlar yapmayı öğrendiniz.

## [İsteğe Bağlı] Meta veri ekleme

Kurduğumuz RAG, vektör deposundaki tüm belgeleri sorgular. Orada birden fazla yılın bilgisinin olduğunu düşünün. Yıllara veya tarihlere göre filtreyebilsek kullanışlı olurdu, değil mi?

Bunu nasıl yaparız? Vektör deposundaki belgelerin meta veri içerdiğini unutmayın. Eğer tarihi ekleyebilseydik, daha sonra filtrelemek için kullanabilirdik.

İpucu: Meta verilerinizi pipeline'ınızda olabildiğince erken ekleyin. Verileriniz vektör deposunda saklandıktan sonra eklemeye çalışmayın.

👉 `embed_and_store()` fonksiyonunuzu uyarlayın.

In [ ]:
def embed_and_store_fancy(file_path, vector_store, session_date):
    """Load a PDF file, split it into chunks, and store the chunks in a vector store.
    Session_date is added to the metadata of each chunk."""
    pass  # YOUR CODE HERE

    return document_ids

👉 Fonksiyonunuzu deneyin ve vektör deponuzun ek meta veri içerdiğini kontrol edin.

In [ ]:
# YOUR CODE HERE

Şimdi alıcıyı kullanıcının sorduğu tarihe göre sınırlamamız gerekiyor.

👉 `answer()` fonksiyonunuzu bir tarih alabilecek ve yeni meta verilere dayalı olarak belgeleri filtreleyebilecek şekilde uyarlayın.

In [ ]:
# YOUR CODE HERE

In [ ]:
# YOUR CODE HERE

Harika! Güçlü bir RAG sistemi oluşturmak için benzerlik aramasını meta veri aramasıyla birleştirdiniz!